# BAN6800 Module 4 — run everything in Google Colab
**Abdi Farah Ahmed · JJUSHYCSH referral no-show risk model**

This notebook runs the **Module 3 pipeline** and then the **Module 4 pipeline** on the real Kaggle file, and gives you every deliverable plus the MLflow UI for screenshots. Runtime: about 20–40 minutes on the free CPU runtime.

Steps: 1) upload the repository zip, 2) upload `KaggleV2-May-2016.csv`, 3) run all cells, 4) take the screenshots, 5) download the outputs zip.

In [ ]:
# 1. Upload the repository zip (jjushycsh-noshow-analytics with the module4 folder inside)
from google.colab import files
import zipfile, os, glob
up = files.upload()
zname = [n for n in up if n.endswith('.zip')][0]
zipfile.ZipFile(zname).extractall('.')
cands = [p for p in glob.glob('**/run_pipeline.py', recursive=True)]
REPO = os.path.dirname(cands[0]) or '.'
print('Repository root:', REPO)
%cd $REPO

In [ ]:
# 2. Upload the Kaggle file (download it from https://www.kaggle.com/datasets/joniarroba/noshowappointments)
from google.colab import files
os.makedirs('data/external', exist_ok=True)
up = files.upload()
csv = [n for n in up if n.lower().endswith('.csv')][0]
os.replace(csv, 'data/external/KaggleV2-May-2016.csv')
!wc -l data/external/KaggleV2-May-2016.csv

In [ ]:
# 3. Install dependencies (Module 3 pandas validator route + Module 4 stack)
!pip -q install -r module4/requirements.txt
import os; os.environ['NOSHOW_HASH_SALT'] = 'colab-run-salt'   # any secret; never commit it

In [ ]:
# 4. Module 3 pipeline: ingest -> validate -> clean -> anonymise -> features -> validate -> publish + bias check
!python run_pipeline.py --skip-gx | tail -40
!ls data/processed reports

In [ ]:
# 5. Module 4: baselines, necessity review, 3 tuned algorithms (MLflow), SHAP, counterfactuals, fairness, mitigation, sensitivity, report
!cd module4 && python run_all.py
!ls module4/outputs module4/outputs/figures

In [ ]:
# 6. MLflow UI for screenshots (runs table, one run's params/metrics, Models -> jjushycsh_noshow_risk)
import subprocess, time
proc = subprocess.Popen(['mlflow', 'ui', '--backend-store-uri', 'sqlite:///module4/mlflow.db', '--port', '5000'])
time.sleep(8)
from google.colab import output
output.serve_kernel_port_as_window(5000)   # click the link that appears

In [ ]:
# 7. FastAPI: start the service, call /predict, and open /docs for the screenshot
import subprocess, time, json, requests
api = subprocess.Popen(['uvicorn', 'api.main:app', '--port', '8000'], cwd='module4')
time.sleep(10)
body = json.load(open('module4/api/sample_request.json'))
r = requests.post('http://127.0.0.1:8000/predict', json=body)
print(r.status_code); print(json.dumps(r.json(), indent=2))
from google.colab import output
output.serve_kernel_port_as_window(8000, path='/docs')

In [ ]:
# 8. Execute the SHAP notebook so it is saved with outputs
!cd module4 && jupyter nbconvert --to notebook --execute --inplace notebooks/shap_analysis.ipynb --ExecutePreprocessor.timeout=1200

In [ ]:
# 9. Download everything you need for the submission
!cd module4 && zip -qr ../module4_outputs.zip outputs notebooks mlflow.db mlruns 2>/dev/null; ls -la module4_outputs.zip
from google.colab import files
files.download('module4_outputs.zip')

## After downloading
1. Open `outputs/BAN6800_Module4_Model_Validation_Report_Abdi_Farah_Ahmed.docx`, complete the **yellow** paragraphs, paste the three screenshots and your GitHub link, and check the printed body word count stays within 1,800–2,200.
2. Commit `module4/` (with `outputs/` and the executed notebook, without `mlflow.db`/`mlruns` if large) to the same repository as Module 3.
3. Complete the AI Disclosure Form.